# Предсказание активности соединений против вируса

## От простого baseline к финальной RF + direct SI модели

В ноутбуке собрана **основная траектория решения**, а не полный журнал всех экспериментов:

1. загружаем данные и проводим короткий EDA;
2. фиксируем безопасную предобработку и метрику соревнования;
3. строим простой линейный baseline;
4. отдельно улучшаем самый проблемный таргет `SI`, предсказывая его напрямую;
5. усиливаем `IC50` и `CC50` через Random Forest с отбором признаков;
6. формируем итоговый submission, полностью генерируемый этим ноутбуком (`public score = 267.64645`).

**Метрика соревнования:**

$$
\operatorname{score} =
\frac{
\operatorname{RMSE}(\mathrm{IC50}) +
\operatorname{RMSE}(\mathrm{CC50}) +
\operatorname{RMSE}(\mathrm{SI})
}{3}
$$

**Ограничения решения:** `SI` предсказывается отдельной моделью. Мы не вычисляем его как `CC50 / IC50`, не используем test-targets и не подгоняем значения вручную.

Исходные CSV не включены в public-репозиторий, так как являются закрытыми данными лаборатории; порядок получения данных описан в `README.md`.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
SUBMISSION_COLS = ["IC50", "CC50", "SI"]
TARGET_TO_SUBMISSION = dict(zip(TARGET_COLS, SUBMISSION_COLS))

**Промежуточный комментарий.** Используем только нужные для финальной истории библиотеки: линейный baseline, `GradientBoostingRegressor` для прямого `SI` и `RandomForestRegressor` для `IC50/CC50`. Тяжелые модели, которые не улучшили public score, здесь повторно не запускаются.

In [ ]:
def find_data_dir():
    candidates = [
        Path("."),
        Path(".."),
        Path("/content"),
        Path("/content/drive/MyDrive/kaggle_hackathon"),
    ]
    for candidate in candidates:
        if all((candidate / name).exists() for name in ["train.csv", "test.csv", "sample_submission.csv"]):
            return candidate.resolve()
    raise FileNotFoundError(
        "Не найдены train.csv, test.csv и sample_submission.csv. "
        "В Colab загрузите их в /content или подключите Google Drive."
    )


DATA_DIR = find_data_dir()
FINAL_SUBMISSION_PATH = DATA_DIR / "final_submition.csv"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

feature_cols = [c for c in test.columns if c != "index"]

print("DATA_DIR:", DATA_DIR)
print("train:", train.shape, "| test:", test.shape, "| sample_submission:", sample_submission.shape)
print("Количество молекулярных признаков:", len(feature_cols))
display(train.head(3))

**Промежуточный комментарий.** В данных есть `751` объект train и `250` объектов test; `index` нужен только для формирования submission. Все остальные test-столбцы являются числовыми молекулярными дескрипторами.

In [ ]:
eda_summary = pd.DataFrame({
    "target": TARGET_COLS,
    "missing": [train[t].isna().sum() for t in TARGET_COLS],
    "mean": [train[t].mean() for t in TARGET_COLS],
    "median": [train[t].median() for t in TARGET_COLS],
    "q90": [train[t].quantile(0.90) for t in TARGET_COLS],
    "q98": [train[t].quantile(0.98) for t in TARGET_COLS],
    "max": [train[t].max() for t in TARGET_COLS],
})

missing_features = train[feature_cols].isna().sum()
constant_cols = [c for c in feature_cols if train[c].nunique(dropna=False) <= 1]
duplicate_feature_cols = train[feature_cols].T.duplicated().sum()
full_duplicate_rows = train.duplicated().sum()

print("Пропуски в feature-части train:", int(missing_features.sum()))
print("Строк train хотя бы с одним пропуском:", int(train[feature_cols].isna().any(axis=1).sum()))
print("Константные признаки:", len(constant_cols))
print("Полные дубли признаков-столбцов:", int(duplicate_feature_cols))
print("Полные дубли строк train:", int(full_duplicate_rows))
display(eda_summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, target in zip(axes, TARGET_COLS):
    ax.hist(np.log1p(train[target]), bins=30, color="#147d73", alpha=0.85)
    ax.set_title(f"log1p({target})")
    ax.set_xlabel("log scale")
plt.tight_layout()
plt.show()

**Промежуточный комментарий.** Таргеты имеют сильный правый хвост, особенно `SI`: медиана мала, а максимум экстремально велик. Поэтому `SI` способен доминировать в ошибке. Для базовых моделей пропуски удобно заменять медианой; в итоговой RF-ветке две train-строки с пропусками удаляются, потому что именно такая рабочая конфигурация дала лучший устойчивый результат.

In [ ]:
def competition_score(y_true: pd.DataFrame, y_pred: np.ndarray) -> tuple[float, dict]:
    # Среднее RMSE по трем таргетам
    rmses = {
        target: mean_squared_error(y_true[target], y_pred[:, i]) ** 0.5
        for i, target in enumerate(TARGET_COLS)
    }
    return float(np.mean(list(rmses.values()))), rmses


def basic_feature_matrices(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # Оговоренный командой EDA пайплайн: удаление индекса, констант и полных дублей; заполение пропусков - медианой
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    constants = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
    X_train = X_train.drop(columns=constants)
    X_test = X_test.drop(columns=constants)

    duplicated = X_train.T.duplicated()
    kept_columns = X_train.columns[~duplicated].tolist()
    X_train = X_train[kept_columns]
    X_test = X_test[kept_columns]

    imputer = SimpleImputer(strategy="median")
    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=kept_columns)
    X_test = pd.DataFrame(imputer.transform(X_test), columns=kept_columns)
    return X_train, X_test, constants, int(duplicated.sum())


X_basic, X_test_basic, removed_constants, removed_duplicates = basic_feature_matrices(train, test)
print("Базовый feature matrix:", X_basic.shape)
print("Удалено констант:", len(removed_constants), "| удалено дубликатов колонок:", removed_duplicates)

**Промежуточный комментарий.** Это общий стартовый EDA-пайплайн нашей команды: `index` не используется как признак, пропуски заполняются медианой, константы и полные дубли признаков удаляются. Важная дисциплина: любые преобразования обучаются только на train.

## Карта ключевых экспериментов

Мы двигались крупными гипотезами: сначала установили baseline, затем локализовали самый проблемный таргет, после этого усиливали только оставшиеся компоненты. Public scores ниже взяты из выполненных отправок.

| Блок | Что проверяли | Что узнали | Лучший скор |
|---|---|---|---:|
| Линейный baseline | Ridge, Lasso, ElasticNet, BayesianRidge; raw / `log1p` targets | Линейной зависимости недостаточно; логирование не спасает raw-RMSE | `324.4` |
| Нелинейный baseline | ExtraTrees / target-wise варианты | Деревья дают заметный шаг, но главный источник ошибки еще не найден | `305.63` |
| Фокус на `SI` | RF-log, direct GBR Huber, quantile/winsorized, LGBM/CatBoost варианты | Прямой робастный `SI` - крупнейшее улучшение решения | `271` |
| Фокус на `IC50/CC50` | Multi-output RF, отдельные модели, разные feature-selection thresholds | Совместный RF после отбора признаков оказался сильнее замен отдельных targets | `~269.5` |
| Устойчивая RF-ветка | RF parameters, число деревьев, feature selection | `RF800` дал сильную стабильную промежуточную конфигурацию | `269.16` |
| Итоговая отправка | Зафиксированный RF-вариант после проверки вариативности + direct `SI` | Файл полностью создается запуском ноутбука | `267.64645` |

Важное ограничение: математическая связь между targets не использовалась для вычисления `SI`; он всегда предсказывался напрямую.

## 1. Линейный baseline

Первый разумный ориентир: отдельная Ridge-модель на каждый таргет. Она быстрая, воспроизводимая и показывает, насколько задача вообще линейно объяснима молекулярными дескрипторами.

In [ ]:
def ridge_baseline_submission(X_train, X_test, train_df, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(train_df), len(TARGET_COLS)))
    test_pred = np.zeros((len(X_test), len(TARGET_COLS)))

    for i, target in enumerate(TARGET_COLS):
        y = train_df[target].to_numpy()
        for fit_idx, valid_idx in kf.split(X_train):
            model = Pipeline([
                ("scale", StandardScaler()),
                ("ridge", Ridge(alpha=10.0)),
            ])
            model.fit(X_train.iloc[fit_idx], y[fit_idx])
            oof[valid_idx, i] = model.predict(X_train.iloc[valid_idx])
            test_pred[:, i] += model.predict(X_test) / n_splits

    test_pred = np.maximum(test_pred, 0)
    oof_score, oof_rmse = competition_score(train_df[TARGET_COLS], np.maximum(oof, 0))

    submission = sample_submission.copy()
    submission[SUBMISSION_COLS] = test_pred
    return oof_score, oof_rmse


ridge_oof_score, ridge_oof_rmse = ridge_baseline_submission(
    X_basic, X_test_basic, train
)
print("Ridge OOF score:", round(ridge_oof_score, 3))
print("Ridge OOF RMSE:", {k: round(v, 3) for k, v in ridge_oof_rmse.items()})

**Промежуточный комментарий.** Линейный baseline нужен как нижняя точка отсчета, но его скор (`324.4`) слишком далек от целевого диапазона. Дальнейший подробный перебор `L1/L2` не является главным источником улучшения: необходимы нелинейные модели и отдельная работа с `SI`.

### Что проверялось рядом с линейным baseline

| Эксперимент | Зачем проверяли | Итог |
|---|---|---|
| `Ridge` / `Lasso` / `ElasticNet` / `BayesianRidge` | Установить простой интерпретируемый уровень качества и проверить регуляризацию | Ridge оказался достаточным представителем линейной ветки; класс в целом слаб |
| Raw targets против `log1p` | Таргеты сильно скошены вправо | В log-пространстве модель выглядит стабильнее, но после обратного преобразования raw-RMSE не улучшается |
| Constants / duplicate columns / low variance / correlation pruning | Сократить шум и мультиколлинеарность | Базовая чистка полезна; агрессивное сокращение признаков само по себе большого выигрыша не дает |

**Вывод этапа.** Нужен не тонкий подбор линейной регуляризации, а нелинейная модель и отдельная стратегия для тяжелохвостого `SI`.

## 2. Главная веха: прямое предсказание `SI`

`SI` - наиболее сложный таргет: его медиана мала, а редкие значения на порядки больше основной массы. Обычная squared-error модель начинает уделять непропорциональное внимание нескольким экстремальным объектам, а `log1p` меняет приоритет ошибки относительно официальной raw-RMSE.

Мы не вычисляем `SI` через отношение двух других таргетов, а строим отдельную модель. После сравнения прямых моделей выбран `GradientBoostingRegressor(loss="huber")`:

- boosting способен описывать нелинейные зависимости дескрипторов;
- Huber-loss ведет себя как squared loss на обычных ошибках, но снижает влияние редких экстремальных промахов;
- усреднение предсказаний трех fold-ов уменьшает чувствительность небольшой выборки к одному train split.

In [ ]:
def direct_si_predictions(X_train, X_test, y_si, n_splits=3):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y_si))
    test_pred = np.zeros(len(X_test))

    for fit_idx, valid_idx in kf.split(X_train):
        model = GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            random_state=RANDOM_STATE,
        )
        model.fit(X_train.iloc[fit_idx], y_si.iloc[fit_idx])
        oof[valid_idx] = model.predict(X_train.iloc[valid_idx])
        test_pred += model.predict(X_test) / n_splits

    return np.maximum(oof, 0), np.maximum(test_pred, 0)


si_oof, si_direct_pred = direct_si_predictions(X_basic, X_test_basic, train["SI"])
si_rmse = mean_squared_error(train["SI"], si_oof) ** 0.5
print("Direct SI OOF RMSE (raw scale; чувствителен к редким экстремумам):", round(si_rmse, 3))
print("Direct SI test prediction: mean =", round(si_direct_pred.mean(), 3),
      "| max =", round(si_direct_pred.max(), 3))

**Промежуточный комментарий.** Raw OOF для `SI` нестабилен из-за единичных экстремальных объектов, поэтому скор каггла здесь оказался особенно полезен. Замена прежнего `SI` на прямой Huber-GBR стала крупнейшим улучшением: рабочая ветка перешла примерно от `305.63` к `271`. Эта модель далее фиксируется и не рассчитывает `SI` через другие таргеты.

### Эксперименты для `SI`

| Вариант `SI` | Мотивация | Наблюдение |
|---|---|---|
| Linear / Ridge `SI` | Самый простой direct baseline | Недостаточно гибок для нелинейного тяжелохвостого target |
| RF на `log1p(SI)` | Стабилизировать хвост и использовать нелинейность | Дал первый большой выигрыш относительно линейной ветки |
| Direct `GradientBoostingRegressor(loss="huber")` | Предсказывать raw `SI`, но ослабить влияние экстремумов | Лучший рабочий профиль `SI`; зафиксирован в финальной ветке |
| Quantile / winsorized direct `SI` | Проверить, недооцениваем ли умеренно большие значения | Public score ухудшился (`271.86`), ветка закрыта |
| LGBM / CatBoost direct `SI` и blends | Проверить альтернативные boosting biases | Не улучшили зафиксированный Huber-GBR профиль |

**Почему это оригинальная часть решения.** Мы не просто выбрали более мощную модель, а разделили задачу по поведению targets: для `SI` нужна робастная direct-модель, тогда как для концентрационных показателей далее лучше работает лес.

## 3. Усиливаем `IC50` и `CC50`: Random Forest + feature selection

После фиксации прямого `SI` оставшийся резерв искали только в `IC50` и `CC50`, не меняя успешную SI-ветку. Рабочая конфигурация:

- из train удаляются только две строки с пропусками;
- числовые признаки масштабируются в рамках train-пайплайна;
- отдельный `RandomForestRegressor(n_estimators=300)` на внутреннем split ранжирует признаки;
- оставляются признаки, дающие `85%` cumulative importance;
- итоговая multi-output RF-модель одновременно предсказывает `IC50` и `CC50`.

Почему такая конструкция разумна: оба показателя относятся к концентрациям и имеют более близкую природу, чем `SI`; общий лес может использовать общие структурные сигналы молекулы, не заставляя робастную SI-модель подстраиваться под другой масштаб ошибки.

In [ ]:
def prepare_rf_branch(train_df: pd.DataFrame, test_df: pd.DataFrame, importance_threshold=0.85):
    rf_train = train_df.dropna().reset_index(drop=True)
    y_rf = rf_train[["IC50, mM", "CC50, mM"]].to_numpy()

    X_rf = rf_train[feature_cols].copy()
    X_test_rf = test_df[feature_cols].copy()
    constants = [c for c in X_rf.columns if X_rf[c].nunique() <= 1]
    X_rf = X_rf.drop(columns=constants)
    X_test_rf = X_test_rf.drop(columns=constants)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_rf = scaler.fit_transform(imputer.fit_transform(X_rf))
    X_test_rf = scaler.transform(imputer.transform(X_test_rf))

    selector_fit_idx, _ = train_test_split(
        np.arange(len(X_rf)), test_size=0.20, random_state=RANDOM_STATE
    )
    selector = RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    )
    selector.fit(X_rf[selector_fit_idx], y_rf[selector_fit_idx])

    importance_order = np.argsort(selector.feature_importances_)[::-1]
    cumulative = np.cumsum(selector.feature_importances_[importance_order])
    n_selected = int((cumulative < importance_threshold).sum() + 1)
    selected_idx = importance_order[:n_selected]

    return X_rf[:, selected_idx], X_test_rf[:, selected_idx], y_rf, n_selected, len(rf_train)


X_rf, X_test_rf, y_rf, n_selected, n_rf_rows = prepare_rf_branch(train, test)
print("Train rows in RF branch:", n_rf_rows, "из", len(train))
print("Selected RF features:", n_selected)

**Промежуточный комментарий.** В успешной RF-ветке остаются `749` train-строк и `65` наиболее важных признаков. Это небольшое отличие от начального median-imputation EDA: оно фиксируется здесь явно, потому что именно эта конфигурация воспроизводит рабочую ветку `IC50/CC50`.

In [ ]:
def predict_iccc_rf(X_train, X_test, y_train, n_estimators, model_seed):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=model_seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return np.maximum(model.predict(X_test), 0)


iccc_rf800_pred = predict_iccc_rf(
    X_rf, X_test_rf, y_rf, n_estimators=800, model_seed=RANDOM_STATE
)

stable_submission = sample_submission.copy()
stable_submission[["IC50", "CC50"]] = iccc_rf800_pred
stable_submission["SI"] = si_direct_pred
print("Собран чистый устойчивый пайплайн")
display(stable_submission.head())

**Промежуточный комментарий.** RF-ветка с `300` деревьями сначала дала около `269.5`, а увеличение числа деревьев до `800` улучшило промежуточный результат до **`269.16`**. Эта ветка показала, что выбранные признаки и direct `SI` работают; ниже фиксируется единственный итоговый RF-вариант, который дал еще лучший воспроизводимый score.

### Как пришли к параметрам RF-ветки

| Проверка | Зачем | Результат / решение |
|---|---|---|
| Separate против multi-output моделей для `IC50/CC50` | Понять, полезно ли совместное обучение связанных concentration targets | Рабочей стала multi-output RF-ветка |
| Пороги feature importance и alternate feature pools | Убрать шум среди большого числа дескрипторов | Порог `85%` дал наиболее удачную рабочую конфигурацию (`65` признаков) |
| Замена одного target на LGBM / GBR / ExtraTrees | Найти локальное улучшение одного компонента score | Public score ухудшался; замены закрыты |
| `RF300` против `RF800` | Снизить дисперсию ансамбля деревьев без смены модели | Переход примерно `269.5 -> 269.16` |
| `min_samples_leaf`, `max_features`, новый feature selection | Проверить регуляризацию и устойчивость | Public результаты хуже (`270+` / `271.58`) |
| Target balancing / tail weighting / upper-quantile RF / two-stage tail model | Исправить возможное недопредсказание больших concentration values | Санитарный CV или public feedback не подтвердили выигрыш |

**Вывод этапа.** Финальная конфигурация является не случайным набором параметров, а победителем последовательной проверки: фиксируем удачный `SI`, затем оставляем наиболее устойчивую RF-схему для двух concentration targets.

## 4. Финальная воспроизводимая отправка

В рамках одной проверки чувствительности Random Forest к `random_state` была зафиксирована конфигурация с тем же отбором признаков и `random_state=101` финального `RF300`. Дальнейший перебор seeds не проводился; конфигурация заново запущена полным pipeline.

Финальная конфигурация модели была проверена на Kaggle и получила public score **`267.64645`**. Файл `final_submition.csv` воспроизводимо создается полным запуском этого ноутбука; систематический подбор seeds как стратегия дальнейшего улучшения не использовался.

In [ ]:
final_iccc_pred = predict_iccc_rf(
    X_rf, X_test_rf, y_rf, n_estimators=300, model_seed=101
)

final_submission = sample_submission.copy()
final_submission[["IC50", "CC50"]] = final_iccc_pred
final_submission["SI"] = si_direct_pred
# Убираем различия последних машинных знаков от параллельного исполнения RF.
final_submission[SUBMISSION_COLS] = final_submission[SUBMISSION_COLS].round(8)
final_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("Итоговый submission создан:", FINAL_SUBMISSION_PATH)
print("Public score зафиксированной конфигурации модели: 267.64645")

**Промежуточный комментарий.** Это итоговый submission проекта: `final_submition.csv` заново строится из train при полном запуске ноутбука и использует прямую модель `SI`. Зафиксированная конфигурация показала public score **`267.64645`**.

## 5. Ветки, которые проверили и закрыли

Код всех отрицательных экспериментов намеренно не запускается в финальном ноутбуке: это делает воспроизведение быстрым и не смешивает итоговый pipeline с поисковым журналом. При этом результаты фиксируются для обоснования выбора модели.

| Группа экспериментов | Что проверяли | Почему идея была разумной | Почему не вошло в финал |
|---|---|---|---|
| Линейные модели | Ridge, Lasso, ElasticNet, BayesianRidge | Быстрый интерпретируемый baseline | Слишком высокий score |
| Target transform | Raw и `log1p` targets | Правые хвосты всех targets | Не улучшило официальный raw-RMSE |
| Feature filtering | Constants, duplicates, low variance, correlation pruning | Уменьшить шум среди 210 descriptors | Базовая чистка оставлена, агрессивные фильтры не улучшили public |
| Tree/boosting families | RF, ExtraTrees, GBR, HistGBR, LightGBM, CatBoost, XGBoost | Разные нелинейные inductive biases | RF + direct SI оказался сильнее; XGBoost не добавил OOF-ценности |
| Модели `SI` | RF-log, Huber GBR, quantile/winsorized, LGBM/CatBoost | Главный тяжелохвостый target | Зафиксирован direct Huber-GBR; остальные ухудшали результат |
| Per-target replacements | Отдельно заменяли `IC50` или `CC50` | Локализовать резерв score | Замены LGBM/GBR/ExtraTrees проиграли |
| Blending | RF/CatBoost и другие blends | Разные ошибки моделей могут компенсироваться | OOF-сигнал не перенесся на public (`~271`) |
| Tail-oriented идеи | Target balancing, tail weights, upper-quantile RF, two-stage tail model | Raw-RMSE может зависеть от редких больших values | Не подтвердились на проверках |
| Small-tabular model | TabPFN | Малый размер train выглядит подходящим | CPU-проверка медленная, CV существенно хуже |

In [ ]:
assert final_submission.columns.tolist() == sample_submission.columns.tolist()
assert final_submission["index"].equals(sample_submission["index"])
assert final_submission[SUBMISSION_COLS].notna().all().all()
assert (final_submission[SUBMISSION_COLS] >= 0).all().all()

print("Проверки итогового submission пройдены.")
print("Файл:", FINAL_SUBMISSION_PATH)
print("Зафиксированный public score модели: 267.64645")

**Итого:**

- Выполнен EDA: удаление `index`, проверка пропусков, констант и дубликатов, анализ тяжелых хвостов targets.
- Сравнены линейные, tree-based и boosting-подходы, raw/log targets, feature filtering и tail-oriented идеи.
- Найдена главная веха решения: прямое робастное предсказание `SI` через Huber-GBR.
- Для `IC50/CC50` выбран multi-output RF-пайплайн с feature selection по cumulative importance.
- Единственный финальный файл `final_submition.csv` полностью генерируется кодом ноутбука; зафиксированная конфигурация модели показала public score **`267.64645`**.

Основной вывод: качество резко выросло после декомпозиции задачи по поведению targets - робастная direct-модель для `SI` и совместный Random Forest для `IC50/CC50`.